In [1]:
import os
os.environ["XLA_FLAGS"] = "--xla_gpu_autotune_level=0"
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0"

In [2]:
import os
import tensorflow as tf

I0000 00:00:1784627794.204588  221949 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
# Force TensorFlow to log which device (CPU or GPU) each operation runs on
# tf.debugging.set_log_device_placement(True)

print("Checking available devices...")
gpus = tf.config.list_physical_devices('GPU')

if not gpus:
    print("❌ No GPU found by TensorFlow! Check your LD_LIBRARY_PATH.")
    exit(1)

# Enable memory growth so TF doesn't crash your GTX 1050 VRAM
try:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✅ Found GPU: {gpus[0].name}")
except RuntimeError as e:
    print(e)

print("\n" + "="*50)
print("RUNNING WORKLOAD ON GPU...")
print("="*50 + "\n")

Checking available devices...
✅ Found GPU: /physical_device:GPU:0

RUNNING WORKLOAD ON GPU...



In [4]:
# Target GPU device context
with tf.device('/GPU:0'):
    # 1. Create dummy array data directly on GPU
    X = tf.random.uniform(shape=(1000, 10), minval=-1.0, maxval=1.0)
    # Simple linear relationship: y = sum(x) > 0
    y = tf.cast(tf.reduce_sum(X, axis=1) > 0, dtype=tf.float32)

    # 2. Build a small neural network
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(32, activation='relu', input_shape=(10,)),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    # 3. Train for a few epochs
    print("Training model...")
    model.fit(X, y, epochs=5, batch_size=32, verbose=1)

    # 4. Make a test prediction
    test_sample = tf.random.uniform(shape=(1, 10))
    prediction = model.predict(test_sample)

print("\n" + "="*50)
print(f"✅ Success! Sample Prediction: {prediction[0][0]:.4f}")
print("="*50)

I0000 00:00:1784627876.045274  221949 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3476 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1050, pci bus id: 0000:01:00.0, compute capability: 6.1
/run/media/umbrelluck/Data/Documents/MAGIC/.venv/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training model...
Epoch 1/5


I0000 00:00:1784627883.611102  222312 service.cc:153] XLA service 0x7ff8e0032e80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1784627883.611117  222312 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce GTX 1050, Compute Capability 6.1 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.25.0)
I0000 00:00:1784627883.671594  222312 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1784627884.069971  222312 cuda_dnn.cc:461] Loaded cuDNN version 92500
I0000 00:00:1784627884.078101  222312 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1344__.9
I0000 00:00:1784627884.685509  222312 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


31/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5383 - loss: 0.6894

I0000 00:00:1784627885.397853  222312 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1344__.9


32/32 ━━━━━━━━━━━━━━━━━━━━ 4s 42ms/step - accuracy: 0.6120 - loss: 0.6740
Epoch 2/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7900 - loss: 0.6146
Epoch 3/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8770 - loss: 0.5301
Epoch 4/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9170 - loss: 0.4219
Epoch 5/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9490 - loss: 0.3088
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step

✅ Success! Sample Prediction: 0.9862
